# 02: Fleet exploratory analysis

1. What is the fleet: States, phases, capacity, coverage?
2. Where does it sit relative to the AS/NZS 4777.2 response thresholds? (as in, do we have enough measurement to check response?)
3. How many intervals survive to the Volt-VAr detection window?
4. **Does the fleet actually do Volt-VAr?** (the question that determines whether conformance scoring in 03_ is measuring a response or measuring noise.)
5. What is the `derating_active` flag?

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
from solar_edge.lib import se_metadata as meta
from solar_edge.lib import se_queries as q
from solar_edge.lib import se_plots as plots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG
params = se_params.PARAMS
display(se_store.store_status(con)[["logical_name", "exists", "size_mb", "n_rows"]])

## 0. Data manifest

Note the `substitution` rows. 
They record what this delivery does **not** have nameplate capacity, a flexible-export flag, irradiance

Their absence is explicit rather than inferred from a default.

In [ ]:
display(contract.manifest(config, params))

## 1. Build or refresh the site dimension

`s_99` is the 99th percentile of observed apparent power, ported from `build_s99_estimates.py`. 

It is an **empirical observation, not a manufacturer rating**.

A site that never reached its inverter limit will have an `s_99` below it, which biases the apparent-limit test toward finding symptoms. 

That is why the quantile is swept in sensitivity.

In [ ]:
# Rebuilds only if se_site was built against a different store.
# Pass force=True to rebuild regardless.
meta.refresh_dimensions(con)

checks = meta.check_site_dimension(con)
display(checks)

meta.assert_checks(checks, label="D4")
display(meta.site_summary(con))

## 2. Fleet composition and coverage

In [ ]:
display(q.fleet_composition(con))

coverage = q.monthly_coverage(con)
display(plots.plot_monthly_coverage(coverage))

capacity = q.capacity_distribution(con)
display(plots.plot_capacity_distribution(capacity))

## 3. Timezone resolution check

The fleet diurnal profile in the AEST analysis frame. It must peak at hour 12. A double
hump or a smeared peak would mean the three states were pooled in mismatched frames.

In [ ]:
diurnal = q.diurnal_profile(con)
display(plots.plot_diurnal_profile(diurnal))

## 4. Where the fleet sits relative to the thresholds

This table sizes every downstream claim.

The **240–253 V band** is where Volt-VAr absorption is required and Volt-Watt has not yet engaged

The **253–258 V overlap** is where both respond at once. No disaggregation applied yet.

In [ ]:
bands = q.voltage_band_occupancy(con, config)
display(bands)

vdist = q.voltage_distribution(con, config)
display(plots.plot_voltage_distribution(vdist))

## 5. Cohort funnel

Attrition from the whole store down to the detection window, in the same spirit as
`fetch_population_funnel` in the Solar Analytics work: never present a final count
without showing what it was drawn from.

In [ ]:
display(q.cohort_funnel(con, config, params))

## 6. Does the fleet do Volt-VAr?

Fleet-scale check:

In the generator convention a conforming inverter **absorbs** reactive power(Q < 0) above 240 V, so the median should slope **downward** with rising voltage.

In [ ]:
signature = q.voltvar_signature(con, config)

for cohort in ("single-phase", "three-phase"):
    print(f"\n{'=' * 90}\n{cohort.upper()}\n{'=' * 90}")
    display(signature[signature.cohort == cohort].drop(columns="cohort")
            .reset_index(drop=True))

In [ ]:
spread = q.voltvar_site_spread(con, config)
display(plots.plot_voltvar_spread(spread))
display(spread.groupby("cohort").tail(4))

### How many inverters actually respond?

In [ ]:
response = q.voltvar_site_response(con, config)
display(plots.plot_site_response_histogram(response))

for cohort, g in response.groupby("cohort"):
    n_resp = int((g.delta_q_kvar < -0.05).sum())
    print(f"{cohort:>13}: {n_resp:>4} of {len(g):>4} sites respond "
          f"({100 * n_resp / len(g):.1f}%)  |  "
          f"median delta {g.delta_q_kvar.median():+.3f} kvar")

response.to_csv(C.ARTEFACT_DIR / "voltvar_site_response.csv", index=False)

### 6a. Magnitude, and the two ways of normalising it

**median |Q|** — the *numerator*. How much reactive power the inverter moves, regardless
of direction. Rising with voltage means it is doing more.

**median power factor** — cos(φ) = P / S = P / √(P² + Q²). This is the correct
normalised quantity, and it is bounded in [0, 1]: 1.000 means the inverter is moving no
reactive power at all.

**median tan(φ) = Q/P** — reported in the table as `med_tan_phi`. This is *not* power
factor. An earlier version of this notebook called it that, which was wrong. It is also
confounded, and that confound is worth seeing.

**Why tan(φ) falls for single-phase, even though the response strengthens.** Site voltage
rises *because* export rises, so between 235 V and 250 V:

| | 235 V | 250 V | change |
|---|---|---|---|
| median \|Q\| | 0.154 kvar | 0.229 kvar | **+49%** |
| median P | 1.43 kW | 4.58 kW | **+219%** |
| tan(φ) = Q/P | 0.105 | 0.046 | −56% |

|Q| is *increasing*. tan(φ) falls only because P more than triples underneath it. The
denominator is doing the work.

Power factor is subject to the same arithmetic, but it does not invert the reading: it
moves 0.9934 → 0.9987 over the same span, i.e. **towards unity**, correctly saying the
reactive contribution is small relative to a rapidly growing P. It never suggests the
inverter stopped responding.

The middle panel below shows P explicitly so the confound is visible rather than
inferred. **Read the left panel for the response.**

In [ ]:
character = q.reactive_character(con, config)
display(plots.plot_reactive_character(character))
display(character)

print("Median power factor by cohort:")
display(character.groupby("cohort").med_power_factor.median())

### 6b. Single-phase versus three-phase sign check

In [ ]:
display(q.phase_cohort_comparison(con, config))

## 7. The `derating_active` flag

If it is voltage-driven it corroborates the Volt-Watt / Volt-VAr analysis.

A flat baseline at ordinary voltages is something else.

In [ ]:
derating = q.derating_by_voltage(con, config)
display(plots.plot_derating_by_voltage(derating))
display(derating)

In [ ]:
display(q.derating_by_cohort(con, config))

## 8. Sensitivity of the cohort to its own definition

In [ ]:
variants = {
    "default": config,
    "voltage = mean of phases": config.with_changes(voltage_aggregation="mean"),
    "single-phase only": config.with_changes(phase_cohort="single"),
    "three-phase only": config.with_changes(phase_cohort="three"),
    "exclude derating intervals": config.with_changes(derating_selection="exclude"),
    "include night-anomaly sites": config.with_changes(night_anomaly_selection="include"),
    "sites with >= 300 days": config.with_changes(min_days_observed=300),
}

rows = []
for label, variant in variants.items():
    funnel = q.cohort_funnel(con, variant, params)
    band = funnel.iloc[3]
    rows.append({
        "variant": label,
        "cohort_intervals": int(funnel.iloc[1].n_intervals),
        "in_voltvar_window": int(band.n_intervals),
        "sites_in_window": int(band.n_sites),
    })
display(pd.DataFrame(rows))

## 9. Data-quality report (D7)

Every known issue quantified against an explicit threshold, run against the **built store** .

- **`ok` / `WARN`**: measurable defects with a threshold. These can pass or fail.
- **`STRUCTURAL`**: Properties of the delivery that cannot be fixed and must be carried   as stated limitations: no nameplate, no irradiance, a derating flag with no explicitzero, sparse overnight coverage, timestamps off a common grid.

In [ ]:
from solar_edge.lib import se_diagnostics as diag

quality = diag.data_quality_report(con)
display(quality)

n_warn = int((quality.status == "WARN").sum())
n_struct = int((quality.status == "STRUCTURAL").sum())
print(f"{n_warn} warnings, {n_struct} structural limitations")

quality.to_csv(C.ARTEFACT_DIR / "data_quality_report.csv", index=False)
print(f"Written to {C.ARTEFACT_DIR / 'data_quality_report.csv'}")